# FE747 Data Analytics in Finance
## Market Regime Gauntlet — Simulation Engine

**Instructions:**
1. Run Cell 1 — libraries and Drive mount
2. Run Cell 2 — fetch FF4 + VIX data
3. Run Cell 3 — load asset price CSVs
4. Run Cell 4 — set strategy parameters and click **▶ Run Simulation**
5. Run Cell 6 — export results to Drive

> All cells are collapsed by default. Click ▶ at left to run each cell in order.

In [ ]:
# @title ⚙️ Cell 1 — Libraries & Drive Mount (run once) { display-mode: "form" }
# =============================================================================
# CELL 1 — LIBRARIES + DRIVE MOUNT
# =============================================================================

import io, json, zipfile, re, os
import numpy as np
import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.linear_model import LogisticRegression
from google.colab import drive, files

drive.mount('/content/drive')

DRIVE_PATH       = '/content/drive/My Drive/MSFin_Python/final_project/'
MODELS_PATH     = os.path.join(DRIVE_PATH, 'models/')
SIMULATIONS_PATH = os.path.join(DRIVE_PATH, 'simulations/')
os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(SIMULATIONS_PATH, exist_ok=True)

print("✅ Libraries loaded")
print(f"📁 Drive:        {DRIVE_PATH}")
print(f"📁 Models:       {MODELS_PATH}")
print(f"📁 Simulations:  {SIMULATIONS_PATH}")

In [ ]:
# @title 📡 Cell 2 — FF4 + VIX Data Fetch (run once) { display-mode: "form" }
# =============================================================================
# CELL 2 — FF4 + VIX FETCH
# =============================================================================

def fetch_ff_zip(url):
    hdrs = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html'
    }
    resp = requests.get(url, headers=hdrs, timeout=30)
    resp.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(resp.content))
    csv_name = [n for n in z.namelist() if n.upper().endswith('.CSV')][0]
    raw = z.read(csv_name).decode('latin-1')
    lines = raw.splitlines()
    date_re = re.compile(r'^\s*\d{8}\s*,')
    first_data_idx = None
    for i, line in enumerate(lines):
        if date_re.match(line):
            first_data_idx = i
            header_row_idx = i - 1
            break
    col_header = lines[header_row_idx]
    data_lines = [col_header]
    for line in lines[first_data_idx:]:
        if date_re.match(line):
            data_lines.append(line)
        elif line.strip() == '':
            continue
        else:
            break
    df = pd.read_csv(io.StringIO('\n'.join(data_lines)), index_col=0)
    df.columns = [c.strip() for c in df.columns]
    df.index = pd.to_datetime(df.index.astype(str).str.strip(), format='%Y%m%d')
    df.index.name = 'Date'
    df = df.apply(pd.to_numeric, errors='coerce') / 100
    return df

def fetch_vix():
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text), index_col=0, parse_dates=True)
    df.columns = ['VIX']
    df['VIX'] = pd.to_numeric(df['VIX'], errors='coerce')
    return df['VIX'].dropna()

URL_FF3 = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip'
URL_MOM = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip'

print("Fetching FF3...")
ff3 = fetch_ff_zip(URL_FF3)
print("Fetching Momentum...")
mom = fetch_ff_zip(URL_MOM)
print("Fetching VIX...")
vix = fetch_vix()

ff = ff3.join(mom, how='inner').join(vix, how='inner')
ff = ff.loc['1990-01-01':].dropna()
print(f"✅ Panel: {ff.shape[0]:,} obs  ({ff.index[0].date()} → {ff.index[-1].date()})")

# Feature engineering
MKT_RF_COL = [c for c in ff.columns if 'mkt' in c.lower() or 'rm' in c.lower()][0]
SMB_COL    = [c for c in ff.columns if 'smb'  in c.lower()][0]
HML_COL    = [c for c in ff.columns if 'hml'  in c.lower()][0]
RF_COL     = [c for c in ff.columns if 'rf'   in c.lower() and 'mkt' not in c.lower()][0]
MOM_COL    = [c for c in ff.columns if 'mom'  in c.lower() or 'wml' in c.lower()][0]

ff['Mkt'] = ff[MKT_RF_COL] + ff[RF_COL]
ff['Mkt_idx'] = (1 + ff['Mkt']).cumprod()

RAW_COLS = {
    MKT_RF_COL: 'Mkt_RF', SMB_COL: 'SMB', HML_COL: 'HML',
    MOM_COL: 'Mom', RF_COL: 'RF', 'VIX': 'VIX'
}
for raw_col, nice in RAW_COLS.items():
    ff[f'{nice}_raw'] = ff[raw_col]
    ff[f'{nice}_5d']  = ff[raw_col].rolling(5).mean()
    ff[f'{nice}_21d'] = ff[raw_col].rolling(21).mean()

FEAT_COLS = [c for c in ff.columns if c.endswith(('_raw', '_5d', '_21d'))]
for fc in FEAT_COLS:
    ff[fc] = ff[fc].shift(1)

ff_base = ff.dropna(subset=FEAT_COLS).copy()
print(f"✅ Features ready: {len(FEAT_COLS)} cols, {len(ff_base):,} obs")

In [ ]:
# @title 💰 Cell 3 — Asset Price Data (run once) { display-mode: "form" }
# =============================================================================
# CELL 3 — ASSET CSV READS
# =============================================================================

ASSET_META = {
    'VFINX': {'name': 'Vanguard 500 Index',           'type': 'Equity'},
    'VEXMX': {'name': 'Vanguard Extended Market',      'type': 'Equity'},
    'VWIGX': {'name': 'Vanguard International Growth', 'type': 'Equity'},
    'VUSTX': {'name': 'Vanguard Long-Term Treasury',   'type': 'Fixed Income'},
    'VBMFX': {'name': 'Vanguard Total Bond Market',    'type': 'Fixed Income'},
    'VWEHX': {'name': 'Vanguard High-Yield Corporate', 'type': 'Fixed Income'},
}

EQUITY_TICKERS = [t for t, m in ASSET_META.items() if m['type'] == 'Equity']
FIXED_TICKERS  = [t for t, m in ASSET_META.items() if m['type'] == 'Fixed Income']

def load_price_csv(ticker):
    path = os.path.join(DRIVE_PATH, 'fund_data/', f'{ticker}_daily.csv')
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df.set_index('Date').sort_index()
    df.index = pd.to_datetime(df.index)
    price = pd.to_numeric(df['Close'], errors='coerce').dropna()
    price.name = ticker
    return price

prices = {}
for ticker in ASSET_META:
    try:
        prices[ticker] = load_price_csv(ticker)
        print(f"✅ {ticker}: {len(prices[ticker]):,} obs  "
              f"({prices[ticker].index[0].date()} → {prices[ticker].index[-1].date()})")
    except Exception as e:
        print(f"❌ {ticker}: {e}")

price_df = pd.DataFrame(prices).sort_index()
returns_df = price_df.pct_change()
print(f"\n✅ Price panel ready: {price_df.shape}")


In [ ]:
# @title 🎮 Cell 4 — Strategy Input & Controls { display-mode: "code" }
# =============================================================================
# CELL 4 — INPUT WIDGETS
# =============================================================================

def load_json_models(team_prefix):
    """Load and rehydrate drawdown + upside models from JSON files."""
    dd_path = os.path.join(MODELS_PATH, f'{team_prefix}_drawdown_model.json')
    up_path = os.path.join(MODELS_PATH, f'{team_prefix}_upside_model.json')

    with open(dd_path) as f:
        dd_json = json.load(f)
    with open(up_path) as f:
        up_json = json.load(f)

    # Validate fit windows match
    dd_fw = dd_json['fit_window']
    up_fw = up_json['fit_window']
    if dd_fw != up_fw:
        print(f"⚠️  WARNING: Fit windows do not match for {team_prefix}")
        print(f"   Drawdown: {dd_fw}")
        print(f"   Upside:   {up_fw}")
    else:
        print(f"✅ {team_prefix}: fit windows match ({dd_fw})")

    def rehydrate(model_json, model_key):
        m         = model_json[model_key]
        feat_cols = m['features']
        coef_vec  = np.array([m['coefficients'][f] for f in feat_cols])
        intercept = m['coefficients']['intercept']
        clf = LogisticRegression()
        clf.classes_   = np.array([0, 1])
        clf.coef_      = coef_vec.reshape(1, -1)
        clf.intercept_ = np.array([intercept])
        return clf, feat_cols, float(m['probability_cutoff']), float(m['threshold_pct'])

    clf_dd, feat_dd, cutoff_dd, thresh_dd = rehydrate(dd_json, 'drawdown_model')
    clf_up, feat_up, cutoff_up, thresh_up = rehydrate(up_json, 'upside_model')

    return {
        'prefix':     team_prefix,
        'dd_json':    dd_json,
        'up_json':    up_json,
        'clf_dd':     clf_dd,
        'clf_up':     clf_up,
        'feat_dd':    feat_dd,
        'feat_up':    feat_up,
        'cutoff_dd':  cutoff_dd,
        'cutoff_up':  cutoff_up,
        'thresh_dd':  thresh_dd,
        'thresh_up':  thresh_up,
        'fit_window': dd_fw,
    }

# ── Discover available JSON pairs in Drive folder ─────────────────────────────
json_files   = [f for f in os.listdir(MODELS_PATH) if f.endswith('_drawdown_model.json')]
team_prefixes = sorted([f.replace('_drawdown_model.json', '') for f in json_files])
print(f"Found strategies: {team_prefixes}")

# ── Simulation window ─────────────────────────────────────────────────────────
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

sim_start_year  = widgets.Dropdown(
    options=list(range(1991, 2025)), value=2004,
    description='Start Year:', style={'description_width':'initial'},
    layout=widgets.Layout(width='200px'))

sim_start_month = widgets.Dropdown(
    options=MONTH_NAMES, value='Jan',
    description='Start Month:', style={'description_width':'initial'},
    layout=widgets.Layout(width='200px'))

sim_num_months  = widgets.IntSlider(
    value=36, min=24, max=420, step=1,
    description='# Months:', style={'description_width':'initial'},
    layout=widgets.Layout(width='400px'))

# ── Strategy selectors ────────────────────────────────────────────────────────
def make_strategy_widgets(label, prefix_default, eq_default, fi_default):
    prefix_dd = widgets.Dropdown(
        options=team_prefixes, value=prefix_default if prefix_default in team_prefixes else team_prefixes[0],
        description=f'{label} Team:', style={'description_width':'initial'},
        layout=widgets.Layout(width='300px'))

    eq_dd = widgets.Dropdown(
        options=EQUITY_TICKERS, value=eq_default,
        description='Equity Fund:', style={'description_width':'initial'},
        layout=widgets.Layout(width='200px'))

    fi_dd = widgets.Dropdown(
        options=FIXED_TICKERS, value=fi_default,
        description='Fixed Fund:', style={'description_width':'initial'},
        layout=widgets.Layout(width='200px'))

    # Allocation sliders — equity only; FI = 100 - EQ
    agg_eq   = widgets.FloatSlider(value=0.90, min=0.0, max=1.0, step=0.05,
        description='AGG EQ%:', style={'description_width':'initial'},
        layout=widgets.Layout(width='400px'), readout_format='.0%')
    neut_eq  = widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05,
        description='NEUT EQ%:', style={'description_width':'initial'},
        layout=widgets.Layout(width='400px'), readout_format='.0%')
    conf_eq  = widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05,
        description='CONF EQ%:', style={'description_width':'initial'},
        layout=widgets.Layout(width='400px'), readout_format='.0%')
    def_eq   = widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05,
        description='DEF EQ%:', style={'description_width':'initial'},
        layout=widgets.Layout(width='400px'), readout_format='.0%')

    return dict(prefix=prefix_dd, eq=eq_dd, fi=fi_dd,
                agg=agg_eq, neut=neut_eq, conf=conf_eq, defn=def_eq)

s1 = make_strategy_widgets('Strategy 1',
    prefix_default=team_prefixes[0] if team_prefixes else '',
    eq_default='VFINX', fi_default='VUSTX')

s2 = make_strategy_widgets('Strategy 2',
    prefix_default=team_prefixes[1] if len(team_prefixes) > 1 else team_prefixes[0],
    eq_default='VFINX', fi_default='VBMFX')

run_btn = widgets.Button(description='▶ Run Simulation',
    button_style='success', layout=widgets.Layout(width='200px'))
out     = widgets.Output()

# ── Layout ────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FE747 Data Analytics in Finance")
print("SIMULATION ENGINE")
print("="*60)

display(widgets.HTML("<b>── Simulation Window ──────────────────────────</b>"))
display(widgets.HBox([sim_start_year, sim_start_month, sim_num_months]))

for label, s in [("Strategy 1", s1), ("Strategy 2", s2)]:
    display(widgets.HTML(f"<b>── {label} ─────────────────────────────────────</b>"))
    display(widgets.HBox([s['prefix'], s['eq'], s['fi']]))
    display(s['agg']); display(s['neut']); display(s['conf']); display(s['defn'])

display(run_btn)
display(out)

In [ ]:
# @title ⚙️ Cell 5 — Simulation Engine (run after Cell 4) { display-mode: "form" }
# =============================================================================
# CELL 5 — SIMULATION ENGINE
# =============================================================================

def run_simulation(sim_params, model, ff_data, ret_data, label):
    """
    Run daily simulation for one strategy.
    Returns a DataFrame with one row per trading day.
    """
    clf_dd   = model['clf_dd']
    clf_up   = model['clf_up']
    cutoff_dd = model['cutoff_dd']
    cutoff_up = model['cutoff_up']
    eq_ticker = sim_params['eq']
    fi_ticker = sim_params['fi']

    agg_eq  = sim_params['agg']
    neut_eq = sim_params['neut']
    conf_eq = sim_params['conf']
    def_eq  = sim_params['defn']

    ALLOC_MAP = {
        ('UP1','DD0'): ('Aggressive',  agg_eq),
        ('UP0','DD0'): ('Neutral',     neut_eq),
        ('UP1','DD1'): ('Conflicted',  conf_eq),
        ('UP0','DD1'): ('Defensive',   def_eq),
    }

    # Simulation date range
    start_dt = pd.Timestamp(year=sim_params['start_year'],
                             month=sim_params['start_month'], day=1)
    end_dt   = start_dt + pd.DateOffset(months=sim_params['num_months']) - pd.DateOffset(days=1)

    # Align all data to common trading days
    sim_days = ff_data.loc[start_dt:end_dt].index
    if len(sim_days) == 0:
        raise ValueError(f"No trading days found for simulation window {start_dt} → {end_dt}")

    # Get last trading day BEFORE simulation starts for initial signal
    pre_days = ff_data.loc[:start_dt - pd.DateOffset(days=1)].index
    if len(pre_days) == 0:
        raise ValueError("No data before simulation start to generate initial signal")

    def get_signal(date):
        """Compute P(D), P(U), signals and allocation state for a given date."""
        if date not in ff_data.index:
            return None
        row = ff_data.loc[[date], FEAT_COLS].astype(float).values
        p_dd = float(clf_dd.predict_proba(row)[0, 1])
        p_up = float(clf_up.predict_proba(row)[0, 1])
        s_dd = int(p_dd >= cutoff_dd)
        s_up = int(p_up >= cutoff_up)
        up_key = f'UP{s_up}'
        dd_key = f'DD{s_dd}'
        state, eq_alloc = ALLOC_MAP[(up_key, dd_key)]
        return p_dd, p_up, s_dd, s_up, state, eq_alloc

    # Initial allocation from last day before window
    init_day    = pre_days[-1]
    init_signal = get_signal(init_day)
    _, _, init_s_dd, init_s_up, init_state, init_eq = init_signal

    # Identify month-end trading days within simulation window
    month_ends = ff_data.loc[start_dt:end_dt].groupby(
        [ff_data.loc[start_dt:end_dt].index.year,
         ff_data.loc[start_dt:end_dt].index.month]
    ).apply(lambda x: x.index[-1])

    # Build month-end signal schedule
    # Key: first day of NEXT month → allocation derived from this month-end signal
    signal_schedule = {}
    prev_result = (init_s_dd, init_s_up, init_state, init_eq)

    # Pre-load initial
    signal_schedule[sim_days[0]] = prev_result

    for (yr, mo), me_date in month_ends.items():
        sig = get_signal(me_date)
        if sig is None:
            continue
        _, _, s_dd, s_up, state, eq_alloc = sig
        # Find first trading day of next month
        next_month_start = pd.Timestamp(year=yr, month=mo, day=1) + pd.DateOffset(months=1)
        next_days = ff_data.loc[next_month_start:].index
        if len(next_days) > 0:
            signal_schedule[next_days[0]] = (s_dd, s_up, state, eq_alloc)

    # ── Daily loop ────────────────────────────────────────────────────────────
    rows = []
    active_s_dd, active_s_up, active_state, active_eq = prev_result
    cum_strat = 1.0
    cum_bmk   = 1.0

    eq_ret  = ret_data[eq_ticker].reindex(sim_days)
    fi_ret  = ret_data[fi_ticker].reindex(sim_days)
    bmk_ret = 0.5 * eq_ret + 0.5 * fi_ret

    for date in sim_days:
        # Update active allocation if month boundary
        if date in signal_schedule:
            active_s_dd, active_s_up, active_state, active_eq = signal_schedule[date]
        active_fi = 1.0 - active_eq

        # Today's "would-be" signal if model ran today
        today_sig = get_signal(date)
        if today_sig:
            t_p_dd, t_p_up, t_s_dd, t_s_up, t_state, t_eq = today_sig
        else:
            t_p_dd = t_p_up = np.nan
            t_s_dd = t_s_up = np.nan
            t_state = np.nan
            t_eq = np.nan

        # Returns
        d_eq  = eq_ret.get(date, np.nan)
        d_fi  = fi_ret.get(date, np.nan)
        d_bmk = bmk_ret.get(date, np.nan)
        d_strat = active_eq * d_eq + active_fi * d_fi if not np.isnan(d_eq) else np.nan

        if not np.isnan(d_strat):
            cum_strat *= (1 + d_strat)
        if not np.isnan(d_bmk):
            cum_bmk   *= (1 + d_bmk)

        rows.append({
            f'{label}_date':            date,
            f'{label}_UP_logit_pct':    t_p_up,
            f'{label}_UP_signal':       t_s_up,
            f'{label}_DD_logit_pct':    t_p_dd,
            f'{label}_DD_signal':       t_s_dd,
            f'{label}_wouldbe_state':   t_state,
            f'{label}_wouldbe_EQ_alloc':t_eq,
            f'{label}_active_state':    active_state,
            f'{label}_active_EQ_alloc': active_eq,
            f'{label}_active_FI_alloc': active_fi,
            f'{label}_EQ_ret':          d_eq,
            f'{label}_FI_ret':          d_fi,
            f'{label}_strat_ret':       d_strat,
            f'{label}_bmk_ret':         d_bmk,
            f'{label}_cum_strat':       cum_strat,
            f'{label}_cum_bmk':         cum_bmk,
        })

    return pd.DataFrame(rows).set_index(f'{label}_date')


def on_run(btn):
    with out:
        clear_output()
        print("Running simulation...")

        # Collect params
        sim_p = {
            'start_year':  sim_start_year.value,
            'start_month': MONTH_NAMES.index(sim_start_month.value) + 1,
            'num_months':  sim_num_months.value,
        }

        try:
            m1 = load_json_models(s1['prefix'].value)
            m2 = load_json_models(s2['prefix'].value)
        except Exception as e:
            print(f"❌ JSON load error: {e}")
            return

        p1 = {**sim_p, 'eq': s1['eq'].value, 'fi': s1['fi'].value,
              'agg': s1['agg'].value, 'neut': s1['neut'].value,
              'conf': s1['conf'].value, 'defn': s1['defn'].value}
        p2 = {**sim_p, 'eq': s2['eq'].value, 'fi': s2['fi'].value,
              'agg': s2['agg'].value, 'neut': s2['neut'].value,
              'conf': s2['conf'].value, 'defn': s2['defn'].value}

        try:
            df1 = run_simulation(p1, m1, ff_base, returns_df, 'strat1')
            df2 = run_simulation(p2, m2, ff_base, returns_df, 'strat2')
        except Exception as e:
            print(f"❌ Simulation error: {e}")
            return

        results = df1.join(df2, how='outer')

        # Add metadata columns at front
        meta = pd.DataFrame({
            'sim_date_run':      pd.Timestamp.now().strftime('%Y-%m-%d'),
            'sim_timestamp':     pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
            'sim_start_year':    sim_p['start_year'],
            'sim_start_mon':     sim_start_month.value,
            'sim_num_mons':      sim_p['num_months'],
            'strat1_team':       s1['prefix'].value,
            'strat1_EQ_fund':    s1['eq'].value,
            'strat1_FI_fund':    s1['fi'].value,
            'strat1_AGG_EQ':     s1['agg'].value,
            'strat1_NEUT_EQ':    s1['neut'].value,
            'strat1_CONF_EQ':    s1['conf'].value,
            'strat1_DEF_EQ':     s1['defn'].value,
            'strat2_team':       s2['prefix'].value,
            'strat2_EQ_fund':    s2['eq'].value,
            'strat2_FI_fund':    s2['fi'].value,
            'strat2_AGG_EQ':     s2['agg'].value,
            'strat2_NEUT_EQ':    s2['neut'].value,
            'strat2_CONF_EQ':    s2['conf'].value,
            'strat2_DEF_EQ':     s2['defn'].value,
        }, index=results.index)

        final = pd.concat([meta, results], axis=1)

        # Store globally for Cell 6
        global SIM_RESULTS, SIM_PARAMS
        SIM_RESULTS = final
        SIM_PARAMS  = sim_p

        print(f"✅ Simulation complete: {len(final):,} trading days")
        print(f"   {sim_start_month.value} {sim_p['start_year']} → "
              f"{final.index[-1].strftime('%b %Y')}")
        print(f"Strat1 final wealth: {final['strat1_cum_strat'].iloc[-1]:.4f}  "
              f"Bmk: {final['strat1_cum_bmk'].iloc[-1]:.4f}")
        print(f"Strat2 final wealth: {final['strat2_cum_strat'].iloc[-1]:.4f}  "
              f"Bmk: {final['strat2_cum_bmk'].iloc[-1]:.4f}")
        print("\nSample output (last 5 rows):")
        display(final[['strat1_active_state','strat1_active_EQ_alloc',
                        'strat1_strat_ret','strat1_cum_strat',
                        'strat2_active_state','strat2_active_EQ_alloc',
                        'strat2_strat_ret','strat2_cum_strat']].tail(5))

run_btn.on_click(on_run)

In [ ]:
# @title 💾 Cell 6 — Export Results { display-mode: "code" }
# =============================================================================
# CELL 6 — OUTPUT EXPORT
# =============================================================================

export_btn = widgets.Button(description='💾 Export to Drive',
    button_style='info', layout=widgets.Layout(width='200px'))
export_out = widgets.Output()
display(export_btn)
display(export_out)

def on_export(btn):
    with export_out:
        clear_output()
        try:
            ts  = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
            t1  = SIM_RESULTS['strat1_team'].iloc[0]
            t2  = SIM_RESULTS['strat2_team'].iloc[0]
            fname = f'sim_{t1}_vs_{t2}_{ts}.csv'
            fpath = os.path.join(SIMULATIONS_PATH, fname)
            SIM_RESULTS.to_csv(fpath)
            print(f"✅ Saved to simulations/: {fpath}")
            files.download(fpath)
        except NameError:
            print("❌ Run simulation first (Cell 5 → Run Simulation button)")
        except Exception as e:
            print(f"❌ Export error: {e}")

export_btn.on_click(on_export)